In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import glob
import matplotlib.pyplot as plt
import plotly
import plotly.express as px
import plotly.graph_objs as go
import h5py
import scanpy as sc
import scipy

In [ ]:
sc.set_figure_params(scanpy=True, fontsize=10, dpi_save = 350)

plt.rcParams["savefig.dpi"] = 350

In [ ]:
#reload anndata with topic model results, if necesseary
adata=sc.read_h5ad('/ix/djishnu/peasena/tf_perturbseq/20240206_perturbseq2/results/240401_merged.h5ad')

In [ ]:
sc.tl.leiden(adata,resolution=0.25,random_state=6)

In [ ]:
sc.tl.leiden(adata,resolution=0.6,random_state=6, key_added='leiden_0pt5', flavor="igraph")

In [ ]:

sc.pl.umap(adata, color=['leiden', "PRDM1", "LMO2"], s = 3, show=False, frameon=False, cmap='inferno')

In [ ]:
adata.uns['leiden_colors'] = ['dodgerblue', 'green', 'firebrick', 'black']

In [ ]:
# Create a mapping dictionary for leiden annotations
leiden_mapping = {
    '0': 'ActB',
    '1': 'GC',
    '2': 'PB',
    '3': 'NA'
}

# Create the new 'leiden_annotation' column by mapping the 'leiden' column
adata.obs['cell_type_annotation'] = adata.obs['leiden'].map(leiden_mapping)

In [ ]:
adata_actb=adata[adata.obs['leiden']=='0'].copy()
adata_gc=adata[adata.obs['leiden']=='1'].copy()
adata_pb=adata[adata.obs['leiden']=='2'].copy()

In [ ]:
import scanpy as sc

# subset data, run differential expression analysis, and save results
def analyze_and_export(adata, group_1, group_2, key_prefix, file_prefix, day, cluster):
    # Subset the data
    adata_subset = adata[adata.obs['group'].isin([group_1, group_2])].copy()

    # Perform differential expression analysis
    key_added = f'deg_{key_prefix}'
    sc.tl.rank_genes_groups(
        adata_subset, 
        'group', 
        key_added=key_added,
        method='t-test_overestim_var',
        use_raw=False,
        pts=True
    )

    # results to CSV
    df_de_gene = sc.get.rank_genes_groups_df(adata_subset, group=None, key=key_added)
    filename = f'/ix/djishnu/peasena/tf_perturbseq/20240206_perturbseq2/results/degs/deg_{day}_{file_prefix}_{cluster}_ttest.csv'
    df_de_gene.to_csv(filename, header=True, index=False)

# parameters for analysis
experiments = [
    {'adata': adata_actb, 'groups': [('D4_batf_r2', 'D4_ntc_r2'), ('D4_irf4_r2', 'D4_ntc_r2'), ('D4_irf8_r2', 'D4_ntc_r2'),
                                  ('D4_prdm1_r2', 'D4_ntc_r2'), ('D4_spib_r1', 'D4_ntc_r2'), ('D4_spib_r2', 'D4_ntc_r2')],
     'day': 'd4', 'cluster': 'actb'},
    {'adata': adata_gc, 'groups': [('D6_batf_r2', 'D6_ntc_r2'), ('D6_irf4_r2', 'D6_ntc_r2'), ('D6_irf8_r2', 'D6_ntc_r2'),
                                  ('D6_prdm1_r2', 'D6_ntc_r2'), ('D6_spib_r1', 'D6_ntc_r2'), ('D6_spib_r2', 'D6_ntc_r2')],
     'day': 'd6', 'cluster': 'gc'},
    {'adata': adata_pb, 'groups': [('D4_batf_r2', 'D4_ntc_r2'), ('D4_irf4_r2', 'D4_ntc_r2'), ('D4_irf8_r2', 'D4_ntc_r2'),
                                   ('D4_prdm1_r2', 'D4_ntc_r2'), ('D4_spib_r1', 'D4_ntc_r2'), ('D4_spib_r2', 'D4_ntc_r2')],
     'day': 'd4', 'cluster': 'pb'},
]

# run for each comparison
for exp in experiments:
    adata = exp['adata']
    day = exp['day']
    cluster = exp['cluster']
    
    for group_1, group_2 in exp['groups']:
        tf = group_1.split('_', 1)[1]  # Extract TF name from group_1
        analyze_and_export(adata, group_1, group_2, f"{day}_{tf}", tf, day, cluster)
